# 09 - The judge gate: retries and the low-confidence fallback

The judge scores every final answer. Below `JUDGE_PASS_SCORE` the critique and its `targets` go back to the planner,
which redoes only the weak parts (or only rewrites the answer). After `JUDGE_MAX_RETRIES` retries, the best attempt is
returned with a **low-confidence** warning.

**Scenarios**
1. An **unanswerable** question - an honest "I don't know" should still pass.
2. An **unreachable quality bar** (pass mark 101, one retry) to force the retry loop and show the attempt history.

**Prerequisites:** `.env` with `OPENAI_API_KEY`, `JUDGE_OPENAI_API_KEY` and `SERPER_API_KEY`; the vector stores (`make ingest`); the CRM & Support MCP server (`make mcp-server-up`).

In [ ]:
# Setup: make src/ importable and load the settings from .env
import sys, warnings
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore", message=".*langchain.mcp.*")

from agentic_rag.config import AgentSettings

settings = AgentSettings.load(ROOT)
print(f"Coordinator: {settings.model} ({settings.coordinator_effort}) | specialists: {settings.specialist_model} "
      f"({settings.specialist_effort}) | judge: {settings.judge_model} (pass >= {settings.judge_pass_score})")

In [ ]:
# Start the multi-agent service: SQLite memory + CRM MCP tools + 5 specialists + Coordinator graph
from agentic_rag.graph import CoordinatorService

service = await CoordinatorService.create(settings)
print(service.crm.describe())
print(service.roster())

## 1. Unanswerable question

In [ ]:
result = await service.ask("How many days of pet adoption leave does Aranya provide?", user_id="emp-arun",
                           on_event=print)
result

## 2. Forcing the retry path
A pass mark of **101** can never be reached, so this run is guaranteed to go through the retry loop and end with the low-confidence fallback (`JUDGE_MAX_RETRIES=1` -> two attempts). The question also asks for launch prices, which the manuals do not contain, so the critique has something real to say. Settings are immutable; `dataclasses.replace` creates the strict copy for a second service.

In [ ]:
from dataclasses import replace

strict = await CoordinatorService.create(replace(settings, judge_pass_score=101, judge_max_retries=1))
result = await strict.ask("Compare the battery capacity, fast-charging wattage and launch price in India of the "
                          "Zentrix Z9, Z9 Plus and Z9 Ultra.",
                          user_id="product-analyst", on_event=print)
result

In [ ]:
for a in result.attempts:
    print(f"Attempt {a['attempt']}: {a['summary']}")
    print("  critique:", a["scores"]["critique"][:400])
    print("  targets :", a["scores"]["targets"])
print("\nlow_confidence:", result.final["low_confidence"], "| best attempt:", result.final["best_attempt"])

In [ ]:
await strict.aclose()
await service.aclose()

## What to observe
- "I don't know" is graded as grounded and relevant when the evidence truly lacks the answer.
- On a retry, the plan round says `RETRY after the quality review failed` and adds steps only for the targets.
- When the bar is never met, the best-scoring attempt is returned with a visible low-confidence notice.